<a href="https://colab.research.google.com/github/arkaprava181204/Grooming_AI_Chatbot/blob/main/AIGroomingModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers accelerate bitsandbytes
!pip install -q sentence-transformers faiss-cpu
!pip install -q gradio pypdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 7.3 MB/s eta 0:00:00


In [2]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [3]:
!pip install -U -q huggingface_hub transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 44.5 MB/s eta 0:00:00


In [4]:
from huggingface_hub import notebook_login

notebook_login()

In [5]:
!hf auth whoami

Hint: The `hf-cli` skill is not installed. Run `hf skills add -g --claude` to teach your AI agents how to use the `hf` CLI.
✓ Logged in
  user: Arka-181204


In [6]:
from huggingface_hub import hf_hub_download

file = hf_hub_download(
    repo_id="google/gemma-3-1b-it",
    filename="config.json"
)

print("SUCCESS!")
print(file)

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

SUCCESS!
/root/.cache/huggingface/hub/models--google--gemma-3-1b-it/snapshots/dcc83ea841ab6100d6b47a070329e1ba4cf78752/config.json


In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "google/gemma-3-1b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Model loaded successfully!


In [ ]:
messages = [
    {
        "role": "user",
        "content": "Write a short poem about Hugging Face."
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(response)

Okay, here's a short poem about Hugging Face:

The models bloom, a vibrant hue,
On Hugging Face, a digital view.
From datasets vast to models bright,
A community shines, a guiding light.

For developers, a helpful hand,
With Transformers, across the land.
Connecting minds, a learning spree,
Hugging Face – for all to see. 

---

Do you want me to try a different style or focus? For


In [ ]:
messages = [
    {
        "role": "user",
        "content": "What is grooming?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(response)

Okay, let's break down what grooming is all about! It's more than just a pretty face – it's a comprehensive practice that focuses on maintaining the health, hygiene, and appearance of an animal. Here's a detailed look:

**1. What is Grooming?**

At its core, grooming is the process of:

* **Maintaining cleanliness:** Removing dirt, debris, and loose hair.
* **Promoting health:**  Addressing skin problems, preventing


In [3]:
!pip install -U -q transformers datasets peft accelerate trl

In [9]:
from datasets import Dataset

dataset = Dataset.from_list(training_data)

print(dataset)

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 60
})


In [4]:
!pip install -U "pyarrow>=18,<22" "datasets>=3.0,<5.1"

ERROR: Operation cancelled by user
^C


KeyboardInterrupt: 

In [10]:
def format_example(example):

    messages = [
        {
            "role": "user",
            "content": example["prompt"]
        },
        {
            "role": "assistant",
            "content": example["completion"]
        }
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False
        )
    }

In [11]:
dataset = dataset.map(format_example)

print(dataset[0]["text"])

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

<bos><start_of_turn>user
What does personal grooming actually mean?<end_of_turn>
<start_of_turn>model
Personal grooming is the regular care and maintenance of a person's body, cleanliness, appearance, and presentation. It includes hygiene, hair care, skin care, oral care, nails, clothing, fragrance, and maintaining a neat overall appearance. Good grooming is not about looking expensive or following every beauty trend; it is mainly about cleanliness, health-supportive habits, appropriateness, and presenting yourself in a way that fits the situation.<end_of_turn>



In [12]:
from peft import LoraConfig

In [13]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    bias="none",
    task_type="CAUSAL_LM"
)

In [14]:
from trl import SFTTrainer, SFTConfig

In [15]:
training_args = SFTConfig(
    output_dir="./gemma-java-tutor",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    fp16=True,
    report_to="none"
)

In [16]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=lora_config,
    processing_class=tokenizer
)

print("Trainer created successfully!")

Adding EOS to train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

Trainer created successfully!


In [8]:
!pip install -U -q "torchao>=0.16.0" peft trl transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 15.4 MB/s eta 0:00:00


In [7]:
training_data = [
    {
        "prompt": "What does personal grooming actually mean?",
        "completion": "Personal grooming is the regular care and maintenance of a person's body, cleanliness, appearance, and presentation. It includes hygiene, hair care, skin care, oral care, nails, clothing, fragrance, and maintaining a neat overall appearance. Good grooming is not about looking expensive or following every beauty trend; it is mainly about cleanliness, health-supportive habits, appropriateness, and presenting yourself in a way that fits the situation."
    },

    {
        "prompt": "Why is grooming more than just looking attractive?",
        "completion": "Grooming serves several purposes beyond appearance. Hygiene helps maintain cleanliness and comfort, while appropriate personal care can help prevent some common hygiene-related problems. Grooming also influences how a person presents themselves socially. It should therefore be viewed as routine self-maintenance rather than simply trying to look attractive."
    },

    {
        "prompt": "How should I understand my skin before choosing a skincare routine?",
        "completion": "Start by observing how your skin behaves rather than choosing products only because they are popular. Notice whether your skin tends to become oily, dry, tight, irritated, or relatively balanced. Also consider climate, sweating, sun exposure, shaving, and the products you already use. A basic routine should generally prioritize gentle cleansing, appropriate moisturization, and protection from excessive sun exposure before adding specialized products."
    },

    {
        "prompt": "What is the skin barrier and why is it important?",
        "completion": "The skin barrier is the outer protective system of the skin that helps reduce water loss and protects against environmental irritants. When the barrier is healthy, the skin generally tolerates normal environmental exposure better. Excessive cleansing, aggressive scrubbing, or unnecessary use of irritating products can disrupt it and cause dryness, burning, redness, or sensitivity. Good grooming should support the skin barrier rather than constantly trying to strip the skin."
    },

    {
        "prompt": "Why does sweat sometimes produce body odor?",
        "completion": "Fresh sweat is not necessarily strongly odorous. Body odor commonly develops when substances in sweat interact with microorganisms living on the skin. Warm and moist areas can be particularly prone to odor. Regular washing, clean clothing, appropriate deodorant or antiperspirant use, and keeping clothing and footwear dry can help manage the problem."
    },

    {
        "prompt": "Why should grooming advice be personalized?",
        "completion": "People differ in age, skin type, hair characteristics, lifestyle, climate, occupation, activity level, culture, budget, and personal preferences. A routine that works well for one person may be unnecessary or irritating for another. Good grooming advice should therefore identify the person's actual needs before recommending specific practices."
    }
]

In [8]:
training_data += [

    # ============================================================
    # 1. UNDERSTANDING GROOMING
    # ============================================================

    {
        "prompt": "Why does consistent grooming usually work better than occasional intensive grooming?",
        "completion": "Regular maintenance prevents dirt, odor, overgrown nails, unmanaged hair, poorly maintained clothing, and other small issues from accumulating. Occasional intensive grooming may temporarily improve appearance, but consistent basic care is easier on the body and usually requires less effort overall."
    },

    {
        "prompt": "What should be considered the foundation of a grooming routine?",
        "completion": "The foundation is basic cleanliness, oral care, clean clothing, appropriate hair and nail maintenance, body-odor management, and keeping frequently used personal items reasonably clean. More advanced cosmetic care should come after these fundamentals."
    },

    {
        "prompt": "Why should grooming focus on maintenance rather than perfection?",
        "completion": "Human bodies naturally have skin texture, body hair, pores, wrinkles, different body shapes, and other variations. Grooming should maintain cleanliness, comfort, and presentation rather than creating an unrealistic expectation of physical perfection."
    },

    {
        "prompt": "How can someone identify the most neglected part of their grooming?",
        "completion": "Look systematically at hygiene, oral care, hair, skin, nails, clothing, footwear, odor, and personal-item cleanliness. The goal is to identify areas that are repeatedly overlooked rather than focusing only on the most visible feature."
    },

    {
        "prompt": "Why should grooming habits be adapted to lifestyle?",
        "completion": "A person who exercises daily, works outdoors, travels frequently, spends most of the day in an office, or lives in a humid climate has different grooming demands. A useful routine should respond to actual exposure, activity, and environment."
    },

    {
        "prompt": "How can grooming affect personal comfort?",
        "completion": "Clean skin, maintained nails, appropriate clothing, manageable hair, oral freshness, and dry footwear can reduce discomfort during daily activities. Grooming is therefore partly about physical comfort rather than appearance alone."
    },

    {
        "prompt": "Why should grooming products be chosen based on a problem rather than popularity?",
        "completion": "A popular product may not address an individual's actual needs and can sometimes introduce unnecessary ingredients or irritation. Identifying the problem first makes it easier to choose only the products that have a useful purpose."
    },

    {
        "prompt": "How can someone improve grooming gradually instead of changing everything at once?",
        "completion": "Establish essential hygiene first, then address one additional area at a time. This makes habits easier to maintain and makes it easier to determine whether a new product or practice is actually beneficial."
    },

    {
        "prompt": "Why should grooming recommendations consider personal preferences?",
        "completion": "Grooming is partly personal. A person may prefer short hair, long hair, facial hair, minimal fragrance, certain clothing styles, or no cosmetic products. As long as hygiene, safety, and appropriateness are maintained, the assistant should respect reasonable preferences."
    },

    {
        "prompt": "How should grooming advice change with age?",
        "completion": "Grooming needs evolve with development. Children need simple supervised routines, adolescents may experience increased sweating and skin changes, adults may have different hair, skin, and lifestyle requirements, and older adults may need greater attention to comfort and gentle care."
    },

    {
        "prompt": "Why should grooming advice distinguish between cosmetic concerns and hygiene concerns?",
        "completion": "A cosmetic concern relates mainly to appearance or personal preference, while a hygiene concern involves cleanliness and maintenance. Distinguishing them prevents unnecessary cosmetic treatment from being presented as essential hygiene."
    },

    {
        "prompt": "What makes a grooming routine sustainable?",
        "completion": "A sustainable routine is simple enough to repeat, appropriate for the person's lifestyle, affordable enough to maintain, and based on genuine needs. A theoretically excellent routine is not useful if the person cannot realistically follow it."
    },


    # ============================================================
    # 2. SKIN CARE
    # ============================================================

    {
        "prompt": "Why does skin need both cleansing and protection?",
        "completion": "Cleansing removes accumulated sweat, excess oil, dirt, and product residue, while protection helps preserve the skin's natural barrier against environmental stress. Excessive cleansing without protection can leave skin irritated, while inadequate cleansing can allow buildup."
    },

    {
        "prompt": "Why is hot water not always better for cleaning the skin?",
        "completion": "Very hot water can increase dryness and irritation by affecting the skin's protective oils and barrier. Comfortable, moderate-temperature water is generally sufficient for normal cleansing."
    },

    {
        "prompt": "Why can washing the face with ordinary body soap be unsuitable for some people?",
        "completion": "Different cleansing products are formulated for different purposes, and some body soaps can be relatively harsh for facial skin. If a product causes excessive dryness, tightness, burning, or irritation, a gentler facial cleanser may be more appropriate."
    },

    {
        "prompt": "Why does moisturizing help maintain the skin barrier?",
        "completion": "Moisturizers help reduce water loss and support the outer layer of the skin. Different formulations provide different combinations of humectants, emollients, and occlusive ingredients, so the suitable texture depends on the individual's skin and environment."
    },

    {
        "prompt": "Why can acne-prone skin still need moisturizer?",
        "completion": "Acne-prone skin can still experience dehydration or barrier disruption. Avoiding all moisturization may increase dryness and irritation, especially when using acne treatments. A suitable lightweight moisturizer can support the barrier without necessarily worsening oiliness."
    },

    {
        "prompt": "Why should acne not be treated by constantly scrubbing the face?",
        "completion": "Acne is not simply a result of surface dirt. Aggressive scrubbing can irritate inflamed skin and damage the barrier, potentially making the skin more uncomfortable. Acne management should focus on appropriate skin care rather than trying to scrub away every blemish."
    },

    {
        "prompt": "Why should pimples generally not be squeezed repeatedly?",
        "completion": "Squeezing can increase inflammation and injure surrounding tissue. It can also increase the chance of temporary marks or scarring. Gentle care is preferable, and persistent or severe acne may require professional treatment."
    },

    {
        "prompt": "How does sun exposure affect everyday skin grooming?",
        "completion": "Repeated ultraviolet exposure contributes to tanning, premature skin changes, and cumulative skin damage. Sun protection should therefore be considered part of routine daytime skin maintenance when exposure is significant."
    },

    {
        "prompt": "Why should skincare products around the eyes be used carefully?",
        "completion": "The skin around the eyes is relatively delicate and products can accidentally enter the eyes. Products should be used according to their directions and irritating products should not be applied unnecessarily close to the eyes."
    },

    {
        "prompt": "Why can shaving affect a skincare routine?",
        "completion": "Shaving creates mechanical friction and can temporarily increase skin sensitivity. Cleansing, shaving products, aftercare, and active skincare products may therefore need to be coordinated to avoid unnecessary irritation."
    },

    {
        "prompt": "Why should towels and pillowcases be included in a skin-care routine?",
        "completion": "They regularly contact the skin and can accumulate sweat, oils, product residue, and environmental material. Reasonable laundering helps maintain general cleanliness and complements personal skin care."
    },

    {
        "prompt": "How should someone approach skincare when they have very sensitive skin?",
        "completion": "Keep the routine simple, avoid unnecessary active ingredients and harsh physical treatments, introduce changes cautiously, and monitor for irritation. Persistent or significant skin reactions should be evaluated by an appropriate healthcare professional."
    },


    # ============================================================
    # 3. HAIR & SCALP
    # ============================================================

    {
        "prompt": "Why is conditioner generally applied differently from shampoo?",
        "completion": "Shampoo is primarily intended to cleanse the scalp and hair of accumulated oils and residues, while conditioner is primarily used to improve the feel, manageability, and surface condition of the hair. The exact application depends on hair type and product instructions."
    },

    {
        "prompt": "Why can excessive shampooing make some hair types feel rough?",
        "completion": "Frequent or aggressive cleansing can remove oils that contribute to hair's surface lubrication, especially in hair that is already dry or porous. The appropriate washing frequency depends on scalp needs and hair characteristics."
    },

    {
        "prompt": "Why can brushing hair too frequently cause problems?",
        "completion": "Repeated mechanical friction can contribute to breakage, particularly when hair is fragile, tangled, or handled aggressively. Brushing should serve a useful purpose and should be done gently with a suitable tool."
    },

    {
        "prompt": "How does hair length affect grooming requirements?",
        "completion": "Longer hair generally has more surface area and may require greater attention to detangling, conditioning, drying, and preventing mechanical damage. Shorter hair may require less maintenance but still needs scalp and haircut care."
    },

    {
        "prompt": "Why does regular haircut maintenance matter even when someone wants long hair?",
        "completion": "Long hair still benefits from maintaining its overall shape and managing damaged or excessively split ends when appropriate. The required frequency depends on hairstyle, growth, hair condition, and personal preference."
    },

    {
        "prompt": "How can environmental pollution affect hair?",
        "completion": "Dust and airborne particles can accumulate on hair and scalp, particularly when someone spends substantial time outdoors. Appropriate cleansing can remove buildup, while excessive washing should still be avoided if it causes dryness or irritation."
    },

    {
        "prompt": "Why can sleeping with wet hair be uncomfortable for some people?",
        "completion": "Wet hair can remain damp against the scalp and pillow and may increase friction and tangling. It can also simply be uncomfortable. Allowing hair to dry appropriately before sleep may make hair management easier, particularly for longer or textured hair."
    },

    {
        "prompt": "Why should hair tools be chosen according to hair type?",
        "completion": "Different hair textures and lengths respond differently to brushing, combing, and styling. A tool that works well for straight hair may not be ideal for tightly curled or fragile hair. The goal is effective styling with minimal unnecessary mechanical stress."
    },

    {
        "prompt": "How can tight hairstyles affect hair?",
        "completion": "Repeated tension from very tight hairstyles can place stress on hair and follicles. If a hairstyle causes pain, headaches, scalp tenderness, or repeated breakage, it may be too tight or unsuitable for frequent use."
    },

    {
        "prompt": "Why should scalp buildup not automatically be treated with stronger shampoo?",
        "completion": "Buildup can have multiple causes, including oil, styling products, or skin conditions. Stronger cleansing is not always the solution and can cause dryness or irritation. Persistent scalp problems may require professional evaluation."
    },

    {
        "prompt": "Why can hair appear dull even when it is clean?",
        "completion": "Hair appearance depends on its surface condition, moisture, damage, product buildup, texture, and light reflection. Cleanliness alone does not guarantee shine or smoothness, and trying to force shine through excessive products can create buildup."
    },

    {
        "prompt": "How should someone care for chemically treated hair?",
        "completion": "Chemically treated hair may be more vulnerable to dryness and breakage. Gentle handling, appropriate conditioning, controlled heat use, and following product-specific care instructions can help minimize additional stress."
    },


    # ============================================================
    # 4. ORAL GROOMING
    # ============================================================

    {
        "prompt": "Why is plaque considered important in oral grooming?",
        "completion": "Plaque is a bacterial film that continually develops on teeth. If it is not adequately removed through oral hygiene, it can contribute to dental and gum problems. Regular cleaning is therefore more important than occasional intensive cleaning."
    },

    {
        "prompt": "Why should oral grooming include the gumline?",
        "completion": "Plaque can accumulate around the area where teeth meet the gums. Cleaning along this area carefully helps maintain overall oral hygiene. The technique should be gentle rather than forceful."
    },

    {
        "prompt": "Why can food remain between teeth even after brushing?",
        "completion": "Toothbrush bristles cannot reach every narrow space between teeth. Interdental cleaning methods are designed to address these areas and can complement brushing."
    },

    {
        "prompt": "Why is saliva important for the mouth?",
        "completion": "Saliva helps lubricate the mouth and contributes to its natural protective environment. Reduced saliva can cause dryness and may contribute to discomfort and bad breath. Persistent dry mouth can have multiple causes and may require professional evaluation."
    },

    {
        "prompt": "How can smoking affect oral grooming?",
        "completion": "Smoking can contribute to tooth staining, unpleasant breath, and gum problems and has broader serious health consequences. Grooming products cannot neutralize the underlying risks of tobacco use."
    },

    {
        "prompt": "Why can sugary foods matter for oral grooming?",
        "completion": "Frequent exposure to sugars can contribute to conditions that promote tooth decay. Oral hygiene, dietary habits, and regular dental care work together; brushing alone does not make unlimited sugar exposure harmless."
    },

    {
        "prompt": "Why should dental checkups remain part of grooming?",
        "completion": "Daily grooming helps maintain cleanliness, but professional dental examinations can identify problems that may not be obvious to the person. Preventive dental care complements rather than replaces daily oral hygiene."
    },

    {
        "prompt": "How should someone care for a removable dental appliance?",
        "completion": "Follow the cleaning and storage instructions provided by the dental professional. Keep the appliance clean, avoid inappropriate cleaning chemicals or excessive heat when not recommended, and attend scheduled dental follow-ups."
    },

    {
        "prompt": "Why should oral pain not be hidden with mouthwash?",
        "completion": "Mouthwash may change taste or temporarily improve freshness but does not necessarily address the source of pain. Persistent pain can indicate a dental problem that requires professional assessment."
    },

    {
        "prompt": "How does alcohol consumption relate to oral grooming?",
        "completion": "Alcohol-containing beverages can contribute to dry mouth and may have broader oral-health implications depending on frequency and amount. Good oral grooming cannot eliminate the health risks associated with excessive alcohol consumption."
    },

    {
        "prompt": "Why can morning breath occur even in people with good oral hygiene?",
        "completion": "Saliva production generally decreases during sleep, allowing oral bacteria and compounds to accumulate more noticeably. Morning breath is therefore common, but persistent strong odor throughout the day despite good hygiene deserves professional assessment."
    },

    {
        "prompt": "Why should toothbrushes be kept in a way that allows them to dry?",
        "completion": "A toothbrush that remains continuously damp provides a less hygienic storage environment. Keeping it appropriately exposed to air and avoiding unnecessary contact with contaminated surfaces supports better toothbrush hygiene."
    },


    # ============================================================
    # 5. BODY ODOR & SWEAT
    # ============================================================

    {
        "prompt": "Why does body odor vary between different people?",
        "completion": "Body odor is influenced by sweat production, skin microorganisms, genetics, hormones, environment, clothing, diet, and individual physiology. Therefore, two people following similar hygiene routines may naturally have different odor patterns."
    },

    {
        "prompt": "How does clothing material influence sweating comfort?",
        "completion": "Different fabrics differ in breathability, moisture absorption, drying speed, and heat retention. The most comfortable material depends on climate, activity, and individual sweating patterns."
    },

    {
        "prompt": "Why should sweaty clothing not remain against the skin for long periods?",
        "completion": "Persistent moisture combined with friction can cause discomfort and may contribute to odor or irritation. Changing into dry clothing after substantial sweating can improve comfort and hygiene."
    },

    {
        "prompt": "Why can body odor become stronger in hot weather?",
        "completion": "Heat generally increases sweating, creating more moisture on the skin. This can increase the interaction between sweat and skin microorganisms and can also cause clothing to become damp more quickly."
    },

    {
        "prompt": "How should someone manage body odor during travel?",
        "completion": "Plan for changes in climate and activity, carry basic hygiene supplies, change clothing when necessary, keep footwear reasonably dry, and avoid relying exclusively on fragrance to mask odor."
    },

    {
        "prompt": "Why should towels be dried properly after use?",
        "completion": "A towel that remains damp for long periods can develop an unpleasant smell and becomes less pleasant to use. Allowing it to dry fully between uses is an important part of personal-item hygiene."
    }
]

In [17]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Step,Training Loss
10,3.341618
20,2.848098
30,2.291080
40,2.164736


TrainOutput(global_step=45, training_loss=2.595704205830892, metrics={'train_runtime': 92.4102, 'train_samples_per_second': 1.948, 'train_steps_per_second': 0.487, 'total_flos': 40357954232064.0, 'train_loss': 2.595704205830892, 'entropy': 2.3205964028835298, 'num_tokens': 9597.0, 'mean_token_accuracy': 0.5015410766005516, 'epoch': 3.0})

In [18]:
trainer.save_model("./gemma-groom-tutor-1")
tokenizer.save_pretrained("./gemma-groom-tutor-1")

('./gemma-groom-tutor-1/tokenizer_config.json',
 './gemma-groom-tutor-1/chat_template.jinja',
 './gemma-groom-tutor-1/tokenizer.json')

In [19]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    model,
    "./gemma-groom-tutor-1"
)

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [20]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_id = "google/gemma-3-1b-it"
adapter_path = "/content/gemma-groom-tutor-1"

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype="auto",
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

merged_model = model.merge_and_unload()

output_path = "/content/gemma-groom-tur-1-merged"

merged_model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/gemma-groom-tur-1-merged/tokenizer_config.json',
 '/content/gemma-groom-tur-1-merged/chat_template.jinja',
 '/content/gemma-groom-tur-1-merged/tokenizer.json')

In [21]:
!pip install fastapi uvicorn pyngrok nest-asyncio

In [22]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_PATH = "/content/gemma-groom-tur-1-merged"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded!")

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Model loaded!


In [32]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

from fastapi.middleware.cors import CORSMiddleware

app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://127.0.0.1:5500",
        "http://localhost:5500"
    ],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class ChatRequest(BaseModel):
    message: str


@app.get("/")
def home():
    return {
        "message": "Grooming AI API is running"
    }


@app.post("/chat")
def chat(request: ChatRequest):

    messages = [
        {
            "role": "system",
            "content": """
You are a grooming assistant.

You ONLY answer grooming-related questions.

You can answer questions about:
- Hair care
- Hairstyles
- Beard and moustache
- Shaving
- Skincare
- Facial care
- Body grooming
- Hygiene
- Fragrance
- Grooming products

If the question is not related to grooming, respond exactly:

Sorry, I can only answer grooming-related questions.
"""
        },
        {
            "role": "user",
            "content": request.message
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True
        )

    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True
    )

    return {
        "response": response
    }

In [33]:
from pyngrok import ngrok

ngrok.set_auth_token("33PMBuEjRRaL0nXOSnQ9DXHergO_6cRmxutjcgtaM79L1JwAE")

public_url = ngrok.connect(8000)

print(public_url)

NgrokTunnel: "https://fusiform-cannon-inheritable.ngrok-free.dev" -> "http://localhost:8000"


In [36]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def run_server():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )

server_thread = threading.Thread(
    target=run_server,
    daemon=True
)

server_thread.start()

print("FastAPI server started on port 8000")

FastAPI server started on port 8000


In [37]:
from pyngrok import ngrok

public_url = ngrok.connect(8000)

print("Public API URL:")
print(public_url)

PyngrokNgrokHTTPError: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: Your account may not run more than 5 endpoints over a single ngrok agent session.\nThe endpoints already running on this session are:\ntn_3HxiydcQOg7gJ73JoqcJTgoJnnT, tn_3HxjCsVWNugVGDaOBl0keK1AnLI, tn_3HxjSq4psYPemVyNoNZrK3tc6U2, tn_3HxjTPaMMRcBWBhIYdeOl1SVlWT, tn_3HxkvFmrq53x22L7KR5XiH9oE0T.\nUpgrade to a Pay-as-you-go plan at: https://dashboard.ngrok.com/billing/choose-a-plan?plan=paygo\r\n\r\nERR_NGROK_324\r\n"}}


In [ ]:
messages = [
    {
        "role": "user",
        "content": "How should I understand my skin before choosing a skin care product?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(response)

Okay, let's dive into understanding your skin before choosing a skincare product! It's a really important step, as it can significantly impact your results and avoid irritation. Here’s a breakdown of how to do that, broken down into categories:

**1. Assess Your Skin Type - The Foundation**

This is *the most crucial* starting point. There are four main skin types:

* **Dry Skin:** Feels tight, flaky, and can be itchy. Often dull and lacks radiance.
* **Oily Skin:** Shiny appearance, enlarged pores, prone to blackheads and breakouts.
* **Combination Skin:** A mix of oily and dry areas - typically, the T-zone (forehead, nose, chin) is oily, while the cheeks are drier.
* **Sensitive Skin:** Easily irritated, prone to redness, burning, stinging, or itching.  Often reacts poorly to fragrances, dyes, and certain ingredients.

**How to Determine Your Skin


In [ ]:
messages = [
    {
        "role": "user",
        "content": "How can I look presentable without spending much money?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(response)

Okay, let's focus on clean, neat, well-maintained clothes and simple accessories that look put-together without breaking the bank. Here's a breakdown of tips, categorized by effort level and budget:

**1. Clean & Well-Maintained Clothes (Free or Low Cost):**

* **Clean Clothes:** This is the foundation. Ensure clothes are clean, wrinkle-free, and comfortable. A fresh breeze can make a big difference.
* **Proper Fit:** Clothes that fit well are more flattering and easier to manage.
* **Neutral Colors:** Stick to classic neutral colors (black, grey, navy, beige, white, brown) unless you're deliberately going for a bolder color.
* **Iron/Steam (If Necessary):**  A simple iron or steamer can instantly smooth out wrinkles and make clothes look neater. (If you don’t have one, a quick clean with a slightly damp cloth can suffice.)

**2. Simple


In [38]:
git init

SyntaxError: invalid syntax (2830201818.py, line 1)